# 4.0 — dynamic HMM search (4.1 + 4.2 in one pass)

All the work happens in `hmm_dynamic_functions.py`; this notebook only configures and launches.
Fitting itself is `cross_validate_armodel` / `cross_validate_poismodel` from
`segmentation_functions`, unchanged.

**What is different from 4.1 / 4.2**

| | old | here |
|---|---|---|
| kappa | searched, [0,5,50] or [0..1000] | **fixed at 0** — it changed nothing measurable over the searched range |
| lag grid | fixed `[1,10,20,30]` | doubling `1,2,4,8,...`, capped at the nearest grid point to the ACF 1/e crossing (per session, median 32) |
| selection | `find_2_best_param` on bits, unpaired SD | sequential **paired** increments on **raw** held-out LL |
| decoding | separate notebook, reloads + re-inits | same pass, reuses the fitted params and `my_inputs` |
| saved | all cells + a 1.24 MB copy of the design matrix | all cells + a fingerprint |
| errors | bare `except:`, printed a mouse name | recorded per session in the assessment CSV |

**Why the cap is a modelling choice, not a selection.** At this data size no likelihood
criterion identifies the AR order: held-out CV, AIC and BIC all rise monotonically to
whatever the largest order tested is (with N~2e5 frames and ~1e2 parameters a `k·ln(N)`
penalty is ~1e-5 per frame, while the LL differences are ~1e-3). The ceiling therefore has
to come from outside the likelihood; the argument used is that a filter longer than the
signal's own decorrelation time absorbs structure the latent state should explain.
The segmentation is insensitive to the choice (<2% of frames, all boundary jitter).

In [10]:
import os
os.environ["JAX_PLATFORM_NAME"] = "cpu"
import sys
# these notebooks now live in hmm_diagnostics/intermediate_pipeline/, so the
# shared segmentation_functions.py is two levels up
sys.path.insert(0, os.path.abspath(os.path.join('..', '..')))
import numpy as np
import pandas as pd
import pickle

# the search / assess functions
import hmm_dynamic_functions as H
# still used for building the session list, exactly as in 4.1
from segmentation_functions import idxs_from_files

## Configuration — set your paths and variable here

In [11]:
""""""""""""""""""""""""""""""""""""""""""""""""""""""""""""""""""""""""""
"""""""""""""""""""""""""""""" 'RUN CODE' """"""""""""""""""""""""""""""""
""""""""""""""""""""""""""""""""""""""""""""""""""""""""""""""""""""""""""

prefix = '/home/ines/repositories/representation_learning_variability/'

# ---- WHICH DATA (set these) --------------------------------------------------
data_path = prefix + 'paper-individuality/data/design_matrices/'
# data_path = prefix + 'paper-individuality/data/design_matrices/1_camera_setup/session_1/'
# data_path = prefix + 'paper-individuality/data/design_matrices/1_camera_setup/extra_bwm/'

fps = 60.0          # 60 for the two-camera set, 30 for the 1_camera_setup / training sets

var_interest = ['whisker_me']
# var_interest = ['Lick count']
# var_interest = ['avg_wheel_vel']

# ---- SESSION LIST (same as 4.1) ----------------------------------------------
all_files = os.listdir(data_path)
design_matrices = [item for item in all_files
                   if 'design_matrix' in item and 'standardized' not in item]

sessions_to_exclude = []      # paste your exclusion list here if you want one

filtered_design_matrices = np.array(
    [s for s in design_matrices if s.split('_')[2] not in sessions_to_exclude])
print(f'{len(design_matrices)} -> {len(filtered_design_matrices)} '
      f'(excluded {len(design_matrices) - len(filtered_design_matrices)})')

idxs, mouse_names = idxs_from_files(filtered_design_matrices)

# ---- FITTING PARAMETERS ------------------------------------------------------
num_states = 2
num_train_batches = 5
method = 'prior'          # PoissonHMM.initialize() has no `emissions`, so 'kmeans'
                          # is not available for the lick model
fit_method = 'em'
num_iters = 100           # measured: every fit reached 99.99% of its final log-prob by
                          # iteration 99, and gains from 100->200 are ~1e-6. Adequate,
                          # not generous -- the convergence flag reports any exceptions.
kappa = 0.0               # not searched; see the notes above
alpha = 0.05              # for the paired increment test
n_jobs = 2

# z-score continuous signals, leave counts alone (as in 4.1)
zsc = (var_interest == ['whisker_me'])

# ---- WHERE TO SAVE (set this) ------------------------------------------------
fitting_params = f'{num_train_batches}_{method}_{fit_method}_zsc_{zsc}_dynamic/'
save_path = prefix + 'paper-individuality/data/hmm/grid_search_dynamic/' + fitting_params
csv_path = os.path.join(save_path, f'assessments_{var_interest[0]}.csv')

# 4.2's output format is ALSO written here, so downstream notebooks
# (5_syllable_generation etc.) need no change: they read
#     most_likely_states, _, _ = pickle.load(open(states_filename, "rb"))
states_save_path = prefix + 'paper-individuality/data/hmm/most_likely_states/' + fitting_params

print('save_path:', save_path)

342 -> 342 (excluded 0)
save_path: /home/ines/repositories/representation_learning_variability/paper-individuality/data/hmm/grid_search_dynamic/5_prior_em_zsc_True_dynamic/


## Launch

In [12]:
assessments = H.run_all(
    idxs, var_interest, zsc, num_states, num_train_batches, method, fit_method,
    save_path=save_path, data_path=data_path, fps=fps,
    n_jobs=n_jobs, csv_path=csv_path, states_save_path=states_save_path,
    num_iters=num_iters, alpha=alpha, kappa=kappa)

assessments.head()

Found 0 sessions to process.
0 fitted this call; wrote 342 rows -> /home/ines/repositories/representation_learning_variability/paper-individuality/data/hmm/grid_search_dynamic/5_prior_em_zsc_True_dynamic/assessments_whisker_me.csv


""


## Look at the fits

Three independent failure modes — held-out likelihood does **not** predict a degenerate
segmentation (on the lick side `bits_LL` is *anti*-correlated with data quality), so all
of them have to be checked separately.

In [4]:
df = pd.read_csv(csv_path)

print(f'{len(df)} sessions\n')
print('failures / flags:')
for flag in ['collapsed', 'degenerate_occupancy', 'flickering']:
    if flag in df:
        print(f'  {flag:22s} {int(df[flag].sum()):4d}')
print(f"  {'errors':22s} {int((df.error.fillna('') != '').sum()):4d}")
print(f"  {'at the lag cap':22s} {int(df.at_cap.sum()) if 'at_cap' in df else 0:4d}")
print(f"  {'fit_ok':22s} {int(df.fit_ok.sum()):4d} / {len(df)}")

print('\nselected lag:')
print(df.best_lag.value_counts().sort_index().to_string())

print('\nmedian dwell (ms):', round(df.median_dwell_ms.median(), 1))

# the sessions worth looking at by eye
cols = [c for c in ['mouse', 'eid', 'best_lag', 'at_cap', 'median_dwell_ms',
                    'n_segments', 'occupancy_state1', 'bits_LL', 'error'] if c in df]
df[~df.fit_ok][cols]

342 sessions

failures / flags:
  collapsed                 0
  degenerate_occupancy      0
  flickering                7
  errors                    0
  at the lag cap          172
  fit_ok                  335 / 342

selected lag:
best_lag
1       11
2        4
4        2
8      109
16      49
32     133
64      31
128      3

median dwell (ms): 400.0


,mouse,eid,best_lag,at_cap,median_dwell_ms,n_segments,occupancy_state1,bits_LL,error
37,CSH_ZAD_019,7f6b86f9-879a-4ea2-8531-294a221af5d0,1,False,116.666667,28149,0.2867,1.224141,NaN
119,NYU-12,a8a8af78-16de-4841-ab07-fde4b5281a03,1,False,16.666667,71538,0.2209,0.879362,NaN
120,NYU-21,8c33abef-3d3e-4d42-9f27-445e9def08f9,8,True,116.666667,12160,0.2837,3.402560,NaN
163,PL017,1b61b7f2-a599-4e40-abd6-3e758d2c9e25,16,False,33.333333,80751,0.3098,0.680113,NaN
288,ZFM-01936,9b5a1754-ac99-4d53-97d3-35c2f6638507,64,True,150.000000,7144,0.4841,2.280156,NaN
289,ZFM-01936,d0ea3148-948d-4817-94f8-dcaf2342bbbe,64,True,150.000000,8183,0.4850,2.341712,NaN
291,ZFM-01937,113c5b6c-940e-4b21-b462-789b4c2be0e5,32,True,166.666667,8598,0.4894,2.026390,NaN
